In [ ]:
# 📁 Contexto
# No diretório raiz desse documento, existem os três arquivos que serão necessários para a conclusão dessa atividade. Em caso de dúvidas, a pasta de se encontra no desktop dessa máquina na pasta 'pyspark_test'.
# 
# Os dados são fictícios e compreendem uma simulação de um cenário de uma loja de departamentos, para isso temos os arquivos com as seguintes informações:
# 
# users.csv → Dados dos clientes/usuários da loja
# sales.csv → Dados das vendas
# products.json → Dados de cadastro dos produtos
# 
# 
# 📝 Questões
# A atividade consiste nas questões a seguir:
# 
# 1. Declare um novo dataframe que mostre o nome do produto e o valor final da compra.
# 
# 2. Declare um novo dataframe com o valor total gasto por cliente.
# 
# 3. Declare um novo dataframe com os cinco melhores clientes contendo o nome, e-mail e o valor gasto em todo o período.
# 
# 4. Declare um novo dataframe com os cinco produtos mais vendidos nos últimos seis meses (considerando período de dados disponível nos arquivos) contendo o nome do produto e a quantidade de produtos vendidos nesse período.
# 
# 5. Calcular a média de faturamento por cliente e o desvio padrão.
# 
# 6. Classificar os clientes em três categorias: silver, gold, platinum
# 
# platinum: clientes que gastaram mais que a média de faturamento por cliente;
# gold: clientes que gastaram do menor desvio padrão até a média de faturamento por cliente;
# silver: clientes que gastaram no máximo a média menos o desvio padrão do faturamento por cliente;
#
# 7. Salvar um arquivo parquet com os três produtos mais consumidos de cada categoria do cliente.

In [1]:
import requests
import json
import csv
import io
import os
import pandas as pd

In [6]:
URL_products = "https://raw.githubusercontent.com/KaelDucatti/pyspark_test_data/main/products.json"

try:
    response = requests.get(URL_products, timeout=10)
    response.raise_for_status()

    # Correção: o arquivo é uma lista de objetos separados por vírgula,
    # mas sem os colchetes de array — envolvemos manualmente.
    texto = response.text.strip().rstrip(",")   # remove vírgula final, se houver
    dados = json.loads(f"[{texto}]")            # transforma em array JSON válido

    print(f"Status: {response.status_code}")
    print(f"Total de itens: {len(dados)}")
    print(f"\nPrimeiro item:\n{dados[0]}")

except requests.exceptions.HTTPError as e:
    print(f"Erro HTTP: {e.response.status_code} - {e.response.text}")
except requests.exceptions.ConnectionError:
    print("Erro de conexão. Verifique a URL ou sua internet.")
except requests.exceptions.Timeout:
    print("Timeout: a requisição demorou demais.")
except requests.exceptions.RequestException as e:
    print(f"Erro inesperado: {e}")
except json.JSONDecodeError as e:
    print(f"Erro ao parsear JSON: {e}")


Status: 200
Total de itens: 50

Primeiro item:
{'product_id': 1, 'product': 'Camisa Polo', 'price': '$86.50'}


In [ ]:
resposta = dados  

caminho = "data/products.json"

# 1. Lê o que já existe (se o arquivo existir)
if os.path.exists(caminho):
    with open(caminho, "r", encoding="utf-8") as f:
        dados_existentes = json.load(f)
else:
    dados_existentes = []

#print(f"Vagas já salvas: {len(dados_existentes)}")

dados_combinados = dados_existentes + resposta

# 3. Remove duplicatas pelo campo 'product_id'
vistos = set()
dados_unicos = []
for vaga in dados_combinados:
    if vaga['product_id'] not in vistos:
        vistos.add(vaga['product_id'])
        dados_unicos.append(vaga)

# 4. Salva tudo de novo
with open(caminho, "w", encoding="utf-8") as f:
    json.dump(dados_unicos, f, ensure_ascii=False, indent=2)

print(f"Total após atualização: {len(dados_unicos)}")

[{'product_id': 1, 'product': 'Camisa Polo', 'price': '$86.50'}, {'product_id': 2, 'product': 'Calça Jeans', 'price': '$112.11'}, {'product_id': 3, 'product': 'Vestido de Verão', 'price': '$100.19'}, {'product_id': 4, 'product': 'Tênis Esportivo', 'price': '$82.38'}, {'product_id': 5, 'product': 'Camiseta Básica', 'price': '$101.50'}, {'product_id': 6, 'product': 'Saia Midi', 'price': '$108.01'}, {'product_id': 7, 'product': 'Blusa de Manga Longa', 'price': '$100.37'}, {'product_id': 8, 'product': 'Sapatos de Salto Alto', 'price': '$122.81'}, {'product_id': 9, 'product': 'Jaqueta Jeans', 'price': '$116.02'}, {'product_id': 10, 'product': 'Bolsa de Couro', 'price': '$117.15'}, {'product_id': 11, 'product': 'Shorts Jeans', 'price': '$86.50'}, {'product_id': 12, 'product': 'Casaco de Inverno', 'price': '$117.88'}, {'product_id': 13, 'product': 'Camisa Social', 'price': '$80.76'}, {'product_id': 14, 'product': 'Blazer Feminino', 'price': '$101.77'}, {'product_id': 15, 'product': 'Sandálias

In [4]:
URL_sales = "https://raw.githubusercontent.com/KaelDucatti/pyspark_test_data/main/sales.csv"

try:
    response = requests.get(URL_sales, timeout=10)
    response.raise_for_status()

    # ✅ csv.DictReader lê cada linha como dicionário usando o cabeçalho
    leitor = csv.DictReader(io.StringIO(response.text))
    dados_csv_sales = list(leitor)

    print(f"Status: {response.status_code}")
    print(f"Total de itens: {len(dados_csv_sales)}")
    print(f"\nPrimeiro item:\n{dados_csv_sales}")

except requests.exceptions.HTTPError as e:
    print(f"Erro HTTP: {e.response.status_code} - {e.response.text}")
except requests.exceptions.ConnectionError:
    print("Erro de conexão. Verifique a URL ou sua internet.")
except requests.exceptions.Timeout:
    print("Timeout: a requisição demorou demais.")
except requests.exceptions.RequestException as e:
    print(f"Erro inesperado: {e}")

Status: 200
Total de itens: 1000

Primeiro item:
[{'sale_id': '1', 'date': '13/06/2022', 'product_id': '30', 'client_id': '223', 'qtde': '2'}, {'sale_id': '2', 'date': '16/11/2022', 'product_id': '23', 'client_id': '175', 'qtde': '2'}, {'sale_id': '3', 'date': '18/08/2022', 'product_id': '29', 'client_id': '184', 'qtde': '2'}, {'sale_id': '4', 'date': '13/03/2022', 'product_id': '31', 'client_id': '194', 'qtde': '1'}, {'sale_id': '5', 'date': '14/12/2022', 'product_id': '13', 'client_id': '221', 'qtde': '2'}, {'sale_id': '6', 'date': '18/01/2022', 'product_id': '26', 'client_id': '206', 'qtde': '1'}, {'sale_id': '7', 'date': '12/10/2022', 'product_id': '28', 'client_id': '123', 'qtde': '2'}, {'sale_id': '8', 'date': '15/06/2022', 'product_id': '39', 'client_id': '269', 'qtde': '2'}, {'sale_id': '9', 'date': '15/02/2022', 'product_id': '36', 'client_id': '179', 'qtde': '3'}, {'sale_id': '10', 'date': '06/12/2022', 'product_id': '41', 'client_id': '148', 'qtde': '1'}, {'sale_id': '11', '

In [5]:
resposta = dados_csv_sales

caminho = "data/sales.csv"

# 1. Lê o que já existe (se o arquivo existir)
if os.path.exists(caminho):
    with open(caminho, "r", encoding="utf-8") as f:
        leitor = csv.DictReader(f)
        dados_existentes = list(leitor)
else:
    dados_existentes = []

print(f"Registros já salvos: {len(dados_existentes)}")

# 2. Combina e remove duplicatas pelo campo 'sale_id'
dados_combinados = dados_existentes + resposta

vistos = set()
dados_unicos = []
for item in dados_combinados:
    if item['sale_id'] not in vistos:
        vistos.add(item['sale_id'])
        dados_unicos.append(item)

# 3. Salva tudo de volta no CSV
campos = dados_unicos[0].keys()  # cabeçalho vem das chaves do dicionário

with open(caminho, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=campos)
    writer.writeheader()
    writer.writerows(dados_unicos)

print(f"Total após atualização: {len(dados_unicos)}")
print(f"Arquivo salvo em: {os.path.abspath(caminho)}")

Registros já salvos: 1000
Total após atualização: 1000
Arquivo salvo em: c:\backup\alan\alan_de_souza\projetos\projeto portifolio\python\tratamento - relacionamento - classificacao de dados\projeto_etl_python\data\sales.csv


In [6]:
URL_users = "https://raw.githubusercontent.com/KaelDucatti/pyspark_test_data/main/users.csv"

try:
    response = requests.get(URL_users, timeout=10)
    response.raise_for_status()

    # ✅ csv.DictReader lê cada linha como dicionário usando o cabeçalho
    leitor = csv.DictReader(io.StringIO(response.text))
    dados_csv_users = list(leitor)

    print(f"Status: {response.status_code}")
    print(f"Total de itens: {len(dados_csv_users)}")
    print(f"\nPrimeiro item:\n{dados_csv_users}")

except requests.exceptions.HTTPError as e:
    print(f"Erro HTTP: {e.response.status_code} - {e.response.text}")
except requests.exceptions.ConnectionError:
    print("Erro de conexão. Verifique a URL ou sua internet.")
except requests.exceptions.Timeout:
    print("Timeout: a requisição demorou demais.")
except requests.exceptions.RequestException as e:
    print(f"Erro inesperado: {e}")

Status: 200
Total de itens: 300

Primeiro item:
[{'client_id': '1', 'name': 'Mitch Kilpatrick', 'email': 'mkilpatrick0@cdc.gov', 'gender': 'M', 'login': 'mkilpatrick0', 'password': 'uE9+F7h5*P'}, {'client_id': '2', 'name': 'Kit Kyncl', 'email': 'kkyncl1@miitbeian.gov.cn', 'gender': 'M', 'login': 'kkyncl1', 'password': 'nZ1>gR%L'}, {'client_id': '3', 'name': 'Marylou Presman', 'email': 'mpresman2@twitter.com', 'gender': 'F', 'login': 'mpresman2', 'password': 'aP2#@KQI'}, {'client_id': '4', 'name': 'Gilberta Andrieu', 'email': 'gandrieu3@4shared.com', 'gender': 'M', 'login': 'gandrieu3', 'password': 'oN2/1oW(IPSNWwoW'}, {'client_id': '5', 'name': 'Tobiah Boughtflower', 'email': 'tboughtflower4@paypal.com', 'gender': 'M', 'login': 'tboughtflower4', 'password': "hD7{HV(oHo'C&P."}, {'client_id': '6', 'name': 'Prent Clifton', 'email': 'pclifton5@cnbc.com', 'gender': 'F', 'login': 'pclifton5', 'password': 'fF4$?1cViZ'}, {'client_id': '7', 'name': 'Katinka Mosedill', 'email': 'kmosedill6@loc.g

In [7]:
resposta = dados_csv_users

caminho = "data/users.csv"

# 1. Lê o que já existe (se o arquivo existir)
if os.path.exists(caminho):
    with open(caminho, "r", encoding="utf-8") as f:
        leitor = csv.DictReader(f)
        dados_existentes = list(leitor)
else:
    dados_existentes = []

print(f"Registros já salvos: {len(dados_existentes)}")

# 2. Combina e remove duplicatas pelo campo 'sale_id'
dados_combinados = dados_existentes + resposta

vistos = set()
dados_unicos = []
for item in dados_combinados:
    if item['client_id'] not in vistos:
        vistos.add(item['client_id'])
        dados_unicos.append(item)

# 3. Salva tudo de volta no CSV
campos = [c for c in dados_unicos[0].keys() if c is not None]  # remove chaves None

with open(caminho, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=campos, extrasaction="ignore")  # ignora campos extras
    writer.writeheader()
    writer.writerows(dados_unicos)

print(f"Total após atualização: {len(dados_unicos)}")
print(f"Arquivo salvo em: {os.path.abspath(caminho)}")

Registros já salvos: 300
Total após atualização: 300
Arquivo salvo em: c:\backup\alan\alan_de_souza\projetos\projeto portifolio\python\tratamento - relacionamento - classificacao de dados\projeto_etl_python\data\users.csv


In [8]:
with open("data/products.json", "r", encoding="utf-8") as f:
    products_json = json.load(f)
df_products = pd.DataFrame(products_json)
df_products.columns

Index(['product_id', 'product', 'price'], dtype='str')

In [9]:
with open("data/sales.csv", "r", encoding="utf-8") as f:
    leitor = csv.DictReader(f)
    sales_csv = list(leitor)

df_sales = pd.DataFrame(sales_csv)
df_sales['sale_id'] = df_sales['sale_id'].astype('int64')
df_sales['date'] = pd.to_datetime(df_sales['date'])
df_sales['product_id'] = df_sales['product_id'].astype('int64')
df_sales['client_id'] = df_sales['client_id'].astype('int64')
df_sales['qtde'] = df_sales['qtde'].astype('int64')

df_sales

C:\Users\Pichau\AppData\Local\Temp\ipykernel_29536\2398916460.py:7: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df_sales['date'] = pd.to_datetime(df_sales['date'])


,sale_id,date,product_id,client_id,qtde
0,1,2022-06-13,30,223,2
1,2,2022-11-16,23,175,2
2,3,2022-08-18,29,184,2
3,4,2022-03-13,31,194,1
4,5,2022-12-14,13,221,2
...,...,...,...,...,...
995,996,2022-01-03,11,166,3
996,997,2022-12-03,18,73,1
997,998,2022-11-22,11,291,1
998,999,2022-10-18,23,134,2


In [10]:
with open("data/users.csv", "r", encoding="utf-8") as f:
    leitor_users = csv.DictReader(f)
    users_csv = list(leitor_users)

df_users = pd.DataFrame(users_csv)
df_users['client_id'] = df_users['client_id'].astype('int64')


df_users

,client_id,name,email,gender,login,password
0,1,Mitch Kilpatrick,mkilpatrick0@cdc.gov,M,mkilpatrick0,uE9+F7h5*P
1,2,Kit Kyncl,kkyncl1@miitbeian.gov.cn,M,kkyncl1,nZ1>gR%L
2,3,Marylou Presman,mpresman2@twitter.com,F,mpresman2,aP2#@KQI
3,4,Gilberta Andrieu,gandrieu3@4shared.com,M,gandrieu3,oN2/1oW(IPSNWwoW
4,5,Tobiah Boughtflower,tboughtflower4@paypal.com,M,tboughtflower4,hD7{HV(oHo'C&P.
...,...,...,...,...,...,...
295,296,Derrek Gaine,dgaine87@diigo.com,M,dgaine87,gW0{8j7DZ$*
296,297,Caresa Baltzar,cbaltzar88@flavors.me,F,cbaltzar88,zC14Wgj
297,298,Lev Tabb,ltabb89@adobe.com,F,ltabb89,kE8<@kwulUS$52v
298,299,Xylia Woods,xwoods8a@last.fm,F,xwoods8a,pH3.\BL2


In [ ]:
# 1. Declare um novo dataframe que mostre o nome do produto e o valor final da compra.
sales_products = pd.merge(df_sales, df_products, left_on='product_id', right_on='product_id', how='inner', suffixes=('_s', '_p'))
sales_products.loc[:, 'valor_final_compra'] = (sales_products['price'].str.replace('$', '', regex=False).astype('float') * sales_products['qtde'])
sales_products[['product','valor_final_compra']].head(3)
sales_products.head()

,product,valor_final_compra
0,Blusa Cropped,214.50
1,Camiseta Estampada,218.38
2,Óculos de Sol,244.54


In [ ]:
# 2. Declare um novo dataframe com o valor total gasto por cliente.
users_sales = pd.merge(df_users, sales_products, left_on='client_id', right_on='client_id', how='inner', suffixes=('_u', '_sp'))
users_sales.head(3)
df_valor_total_cliente = (
    users_sales.groupby(['client_id', 'name'])['valor_final_compra']
    .sum()
    .reset_index()
    .rename(columns={'valor_final_compra': 'valor_total_gasto'})
)

,client_id,name,email,gender,login,password,sale_id,date,product_id,qtde,product,price,valor_final_compra
0,1,Mitch Kilpatrick,mkilpatrick0@cdc.gov,M,mkilpatrick0,uE9+F7h5*P,188,2022-10-07,32,1,Calça Flare,$82.87,82.87
1,1,Mitch Kilpatrick,mkilpatrick0@cdc.gov,M,mkilpatrick0,uE9+F7h5*P,596,2022-02-14,13,2,Camisa Social,$80.76,161.52
2,2,Kit Kyncl,kkyncl1@miitbeian.gov.cn,M,kkyncl1,nZ1>gR%L,36,2022-06-14,10,3,Bolsa de Couro,$117.15,351.45


In [13]:
# 3. Declare um novo dataframe com os cinco melhores clientes contendo o nome, e-mail e o valor gasto em todo o período.
top_5_clientes = (users_sales.groupby(['client_id', 'name', 'email'], as_index=False).agg(valor_gasto_total=('valor_final_compra', 'sum')).sort_values('valor_gasto_total', ascending=False).head(5))

top_5_clientes

,client_id,name,email,valor_gasto_total
124,131,Randa Friedenbach,rfriedenbach3m@paypal.com,2240.51
235,245,Giuditta Blease,gblease6s@friendfeed.com,1902.85
21,24,Cher Higford,chigfordn@issuu.com,1843.35
191,200,Keen Juggings,kjuggings5j@phoca.cz,1795.86
28,31,Alfie Pattlel,apattlelu@discuz.net,1755.60


In [14]:
# 4. Declare um novo dataframe com os cinco produtos mais vendidos nos últimos seis meses (considerando período de dados disponível nos arquivos) contendo o nome do produto e a quantidade de produtos vendidos nesse período.
 
slice_users_sales = users_sales[['date', 'qtde', 'product']].copy()
slice_users_sales['ano_mes'] = slice_users_sales['date'].dt.strftime('%Y-%m')

# Identifica os 6 meses mais recentes disponíveis nos dados
ultimos_6_meses = sorted(slice_users_sales['ano_mes'].unique())[-6:]

# Filtra apenas esses meses
filtro_periodo = slice_users_sales[slice_users_sales['ano_mes'].isin(ultimos_6_meses)]

# Agrupa só por produto, somando a quantidade total no período
top_produtos = (
    filtro_periodo.groupby('product')['qtde']
    .sum()
    .sort_values(ascending=False)
    .head(5)
    .reset_index()
)

top_produtos

,product,qtde
0,Macaquinho,39
1,Blusa de Manga Longa,34
2,Casaco de Inverno,29
3,Blusa de Malha,29
4,Bolsa de Couro,27


In [ ]:
# 5. Calcular a média de faturamento por cliente e o desvio padrão.

faturamento_por_cliente = (
    users_sales.groupby(['client_id', 'name'])['valor_final_compra']
    .sum()
    .reset_index()
    .rename(columns={'valor_final_compra': 'valor_total_gasto'})
)

# média e desvio padrão ENTRE os clientes
media_faturamento = faturamento_por_cliente['valor_total_gasto'].mean()
desvio_padrao_faturamento = faturamento_por_cliente['valor_total_gasto'].std()

print(f"Média de faturamento por cliente: {media_faturamento:.2f}")
print(f"Desvio padrão: {desvio_padrao_faturamento:.2f}")

,client_id,name,media_faturamento,desvio_padrao
0,1,Mitch Kilpatrick,122.20,55.61
1,2,Kit Kyncl,257.90,122.84
2,3,Marylou Presman,213.46,93.33
3,4,Gilberta Andrieu,222.54,109.79
4,5,Tobiah Boughtflower,224.22,NaN


In [16]:
# 6. Classificar os clientes em três categorias: silver, gold, platinum
# 
# platinum: clientes que gastaram mais que a média de faturamento por cliente;
# gold: clientes que gastaram do menor desvio padrão até a média de faturamento por cliente;
# silver: clientes que gastaram no máximo a média menos o desvio padrão do faturamento por cliente;

# Calcular o total gasto por cliente
total_por_cliente = users_sales.groupby('client_id')['valor_final_compra'].sum()

# Calcular média e desvio padrão
media = total_por_cliente.mean()
desvio = total_por_cliente.std()

print(f"Média: {media:.2f}")
print(f"Desvio Padrão: {desvio:.2f}")
print(f"Limite Silver/Gold: {media - desvio:.2f}")
print(f"Limite Gold/Platinum: {media:.2f}")

# Função de classificação
def classificar_cliente(total):
    if total > media:
        return 'platinum'
    elif total >= media - desvio:
        return 'gold'
    else:
        return 'silver'

# Aplicar classificação
classificacao = total_por_cliente.apply(classificar_cliente).reset_index()
classificacao.columns = ['client_id', 'categoria']

# Merge com users_sales para ter o nome do cliente
resultado = users_sales[['client_id', 'name']].drop_duplicates().merge(classificacao, on='client_id')
resultado['total_gasto'] = resultado['client_id'].map(total_por_cliente)

print("\nClassificação dos clientes:")
resultado_final = resultado.sort_values('total_gasto', ascending=False)
resultado_final

Média: 707.26
Desvio Padrão: 414.34
Limite Silver/Gold: 292.92
Limite Gold/Platinum: 707.26

Classificação dos clientes:


,client_id,name,categoria,total_gasto
124,131,Randa Friedenbach,platinum,2240.51
235,245,Giuditta Blease,platinum,1902.85
21,24,Cher Higford,platinum,1843.35
191,200,Keen Juggings,platinum,1795.86
28,31,Alfie Pattlel,platinum,1755.60
...,...,...,...,...
188,197,Richard Ruddick,silver,102.70
284,294,Blayne Sirr,silver,95.66
71,76,Brittne Weeds,silver,85.49
134,141,Zolly Cave,silver,82.64


In [20]:
# 7. Salvar um arquivo parquet com os três produtos mais consumidos de cada categoria do cliente.

sales_com_categoria = df_sales.merge(classificacao, on='client_id', how='left')

sales_completo = sales_com_categoria.merge(
    df_products[['product_id', 'product']],
    on='product_id',
    how='left'
)

consumo_por_categoria_produto = (
    sales_completo
    .groupby(['categoria', 'product'], as_index=False)['qtde']
    .sum()
)

consumo_por_categoria_produto['rank'] = (
    consumo_por_categoria_produto
    .groupby('categoria')['qtde']
    .rank(method='first', ascending=False)
)

top3_por_categoria = (
    consumo_por_categoria_produto[consumo_por_categoria_produto['rank'] <= 3]
    .sort_values(['categoria', 'rank'])
    .drop(columns='rank')
    .reset_index(drop=True)
)

print(top3_por_categoria)

top3_por_categoria.to_parquet('top3_produtos_por_categoria_cliente.parquet', index=False)

  categoria                product  qtde
0      gold      Casaco de Inverno    26
1      gold         Cinto de Couro    26
2      gold          Camisa Xadrez    23
3  platinum   Blusa de Manga Longa    57
4  platinum        Mochila de Lona    52
5  platinum     Calça Cintura Alta    45
6    silver             Macaquinho     7
7    silver  Sapatos de Salto Alto     6
8    silver            Calça Flare     5
